# QuoteDesk: fine-tune Gemma 3 270M (LoRA) for email extraction

Teacher-student distillation: ground-truth-first synthetic emails (labels generated by code) → LoRA on Gemma 3 270M.

**Runs on Kaggle (GPU T4, internet ON) or Colab.** Outputs go to `/kaggle/working` (or `./outputs`):
`adapter/`, `eval.json` (fine-tuned), `base_eval.json` (base + 3-shot baseline), `loss_curve.png`, `run_config.json`.

All data is synthetic; the company (Northbeam Industrial Supply) and every product are fictional.
Gemma is provided under and subject to the Gemma Terms of Use found at ai.google.dev/gemma/terms.

## 1. Parameters

In [ ]:
BASE_MODEL = "unsloth/gemma-3-270m-it"   # same weights as google/gemma-3-270m-it (ungated mirror)
VERSION_TAG = "v1"                        # tag pushed to the HF model repo
EPOCHS = 3
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
BATCH_SIZE = 8
GRAD_ACCUM = 2
WARMUP_RATIO = 0.05
SEED = 3407
MAX_NEW_TOKENS = 384
GEN_BATCH_SIZE = 16
RUN_BASE_BASELINE = True                  # also evaluate the untuned base model with 3 few-shot examples
PUSH_TO_HUB = True                        # needs the HF_TOKEN notebook secret
HF_MODEL_REPO = "your-hf-user/quotedesk-student"
DATASET_DIR = None                        # auto-detected (Kaggle input / Colab / local bundle)
OUTPUT_DIR = None                         # auto: /kaggle/working or ./outputs

## 2. Install (pins follow the official Unsloth Gemma 3 (270M) notebook)

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install rapidfuzz scipy matplotlib

## 3. Locate data, import the shared eval library

In [ ]:
import glob, json, os, sys, time, platform
from pathlib import Path

def find_dataset_dir():
    if DATASET_DIR:
        return Path(DATASET_DIR)
    pats = ["/kaggle/input/*/train.jsonl", "/kaggle/input/*/*/train.jsonl", "/kaggle/input/*/*/*/train.jsonl",
            "/content/quotedesk-data/train.jsonl", "/content/drive/MyDrive/quotedesk-data/train.jsonl",
            "data/kaggle_bundle/train.jsonl", "../data/kaggle_bundle/train.jsonl"]
    for p in pats:
        hits = sorted(glob.glob(p))
        if hits:
            return Path(hits[0]).parent
    raise FileNotFoundError("Attach the quotedesk-data dataset (Kaggle: Add Input) or set DATASET_DIR")

DATA = find_dataset_dir()
OUT = Path(OUTPUT_DIR) if OUTPUT_DIR else (Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs"))
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(DATA))           # the bundle ships evallib/
import evallib
from evallib.prompt import fewshot_messages
from evallib.runner import evaluate, load_split, write_report, student_cost_usd
from evallib.normalize import email_hash

train_rows = load_split(DATA / "train.jsonl")
val_rows = load_split(DATA / "val.jsonl")
exam_rows = load_split(DATA / "exam.jsonl")
exam_hard_rows = load_split(DATA / "exam_hard.jsonl")
fewshot_rows = load_split(DATA / "fewshot.jsonl")
manifest = json.loads((DATA / "manifest.json").read_text()) if (DATA / "manifest.json").exists() else {}

# corrections (auto/manual retrain): train on base train + corrections, never exam data
corrections = load_split(DATA / "corrections.jsonl") if (DATA / "corrections.jsonl").exists() else []
exam_hashes = {email_hash(r["body"]) for r in exam_rows + exam_hard_rows}
for r in train_rows + val_rows + corrections:
    assert email_hash(r["body"]) not in exam_hashes, f"exam leakage: {r.get('id')}"
seen = {email_hash(r["body"]) for r in train_rows}
corrections = [c for c in corrections if email_hash(c["body"]) not in seen]
print(f"data: {DATA}\ntrain={len(train_rows)} (+{len(corrections)} corrections) val={len(val_rows)} "
      f"exam={len(exam_rows)} exam_hard={len(exam_hard_rows)}")

## 4. Tokenizer + sequence length sized to the data

In [ ]:
import numpy as np
from transformers import AutoTokenizer
from unsloth.chat_templates import get_chat_template

tok0 = get_chat_template(AutoTokenizer.from_pretrained(BASE_MODEL), chat_template="gemma3")
def to_text(tokenizer, messages, add_generation_prompt=False):
    return tokenizer.apply_chat_template(messages, tokenize=False,
                                         add_generation_prompt=add_generation_prompt).removeprefix("<bos>")
lens = [len(tok0(to_text(tok0, r["messages"]))["input_ids"]) for r in train_rows + corrections + val_rows]
p99 = int(np.percentile(lens, 99))
MAX_SEQ_LEN = min(2048, int(np.ceil((p99 + 16) / 128) * 128))
print(f"token length p50={int(np.median(lens))} p99={p99} max={max(lens)} -> MAX_SEQ_LEN={MAX_SEQ_LEN}")

## 5. Base model + LoRA

In [ ]:
import torch
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False, load_in_8bit=False, full_finetuning=False)
tokenizer = get_chat_template(tokenizer, chat_template="gemma3")
model = FastModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED, use_rslora=False, loftq_config=None)

## 6. Datasets (chat format → text; loss on the assistant JSON only)

In [ ]:
from datasets import Dataset
def ds(rows):
    return Dataset.from_list([{"text": to_text(tokenizer, r["messages"])} for r in rows])
train_ds = ds(train_rows + corrections).shuffle(seed=SEED)
val_ds = ds(val_rows)
print(train_ds[0]["text"][:600])

## 7. Train (eval loss every epoch)

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

cfg_fields = set(inspect.signature(SFTConfig.__init__).parameters)
cfg_kwargs = dict(
    dataset_text_field="text", per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE, warmup_ratio=WARMUP_RATIO, lr_scheduler_type="linear",
    logging_steps=10, optim="adamw_8bit", weight_decay=0.01, seed=SEED, output_dir=str(OUT / "trainer"),
    report_to="none", save_strategy="no", per_device_eval_batch_size=BATCH_SIZE)
cfg_kwargs["eval_strategy" if "eval_strategy" in cfg_fields else "evaluation_strategy"] = "epoch"
cfg_kwargs["max_length" if "max_length" in cfg_fields else "max_seq_length"] = MAX_SEQ_LEN
trainer_kwargs = dict(model=model, train_dataset=train_ds, eval_dataset=val_ds, args=SFTConfig(**cfg_kwargs))
if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer
trainer = SFTTrainer(**trainer_kwargs)
trainer = train_on_responses_only(trainer, instruction_part="<start_of_turn>user\n",
                                  response_part="<start_of_turn>model\n")
t0 = time.time()
train_result = trainer.train()
TRAIN_SECONDS = round(time.time() - t0, 1)
print(train_result, f"train time {TRAIN_SECONDS}s")

## 8. Loss curve

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
hist = trainer.state.log_history
tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.figure(figsize=(7, 4))
plt.plot(*zip(*tr), label="train loss")
if ev:
    plt.plot(*zip(*ev), "o-", label="val loss (per epoch)")
plt.xlabel("step"); plt.ylabel("loss"); plt.title(f"QuoteDesk student {VERSION_TAG}"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT / "loss_curve.png", dpi=130); plt.show()

## 9. Evaluate the fine-tuned model (val, exam, exam_hard) with the shared eval library

In [ ]:
FastModel.for_inference(model)
tokenizer.padding_side = "left"

@torch.no_grad()
def batch_generate(mdl, prompts):
    outs = {}
    for i in range(0, len(prompts), GEN_BATCH_SIZE):
        chunk = prompts[i:i + GEN_BATCH_SIZE]
        enc = tokenizer([p for _, p in chunk], return_tensors="pt", padding=True).to(mdl.device)
        t0 = time.time()
        gen = mdl.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, use_cache=True,
                           pad_token_id=tokenizer.pad_token_id)
        per = (time.time() - t0) / len(chunk)
        for (rid, _), g in zip(chunk, gen):
            text = tokenizer.decode(g[enc["input_ids"].shape[1]:], skip_special_tokens=True)
            outs[rid] = {"text": text.strip(), "latency_s": per}
    return outs

def run_split(mdl, rows, make_messages):
    prompts = [(r["id"], to_text(tokenizer, make_messages(r), add_generation_prompt=True)) for r in rows]
    outs = batch_generate(mdl, prompts)
    return evaluate(rows, lambda r: {**outs[r["id"]], "cost_usd": None}, progress_every=0)

student_msgs = lambda r: r["messages"][:2]
tuned = {name: run_split(model, rows, student_msgs)
         for name, rows in (("val", val_rows), ("exam", exam_rows), ("exam_hard", exam_hard_rows))}
for k, v in tuned.items():
    m = v["metrics"]
    print(f"{k:10s} intent={m['intent_accuracy']} json={m['json_validity_rate']} item_f1={m['item_f1']} "
          f"qty={m['quantity_accuracy']} missing_f1={m['missing_info_f1']} exact={m['exact_match_rate']}")

## 10. Save adapter + eval.json + run_config.json

In [ ]:
import importlib.metadata as md_
def ver(p):
    try: return md_.version(p)
    except Exception: return None

model.save_pretrained(OUT / "adapter")
tokenizer.save_pretrained(OUT / "adapter")
run_config = {
    "version_tag": VERSION_TAG, "base_model": BASE_MODEL, "epochs": EPOCHS, "learning_rate": LEARNING_RATE,
    "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT, "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM, "warmup_ratio": WARMUP_RATIO, "seed": SEED, "max_seq_len": MAX_SEQ_LEN,
    "max_new_tokens": MAX_NEW_TOKENS, "train_examples": len(train_rows), "corrections": len(corrections),
    "train_seconds": TRAIN_SECONDS, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "platform": "kaggle" if Path("/kaggle").exists() else ("colab" if "COLAB_GPU" in os.environ else platform.node()),
    "dataset_manifest": manifest,
    "versions": {p: ver(p) for p in ("torch", "transformers", "trl", "peft", "unsloth", "unsloth_zoo", "datasets", "accelerate")},
    "final_train_loss": train_result.training_loss,
    "eval_loss_per_epoch": [h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h],
}
(OUT / "run_config.json").write_text(json.dumps(run_config, indent=2))
write_report(OUT, "tuned", f"{BASE_MODEL}+lora:{VERSION_TAG}", tuned,
             extra={"adapter_version": VERSION_TAG, "hardware": run_config["gpu"],
                    "latency_note": "GPU batch latency; CPU latency is measured separately in eval/run_eval.py",
                    "run_config": run_config})
(OUT / "eval.json").write_text((OUT / "tuned.json").read_text())
print("saved:", sorted(p.name for p in OUT.iterdir()))

## 11. (Optional) Base-model baseline: same prompt + 3 few-shot examples

In [ ]:
if RUN_BASE_BASELINE:
    base_model, _ = FastModel.from_pretrained(model_name=BASE_MODEL, max_seq_length=4096,
                                              load_in_4bit=False, load_in_8bit=False, full_finetuning=False)
    FastModel.for_inference(base_model)
    shots = [{"sender": r["sender"], "subject": r["subject"], "body": r["body"], "target": r["target"]} for r in fewshot_rows]
    few = lambda r: fewshot_messages(shots, r["sender"], r["subject"], r["body"])
    base = {name: run_split(base_model, rows, few) for name, rows in (("exam", exam_rows), ("exam_hard", exam_hard_rows))}
    write_report(OUT, "base", f"{BASE_MODEL} (3-shot)", base,
                 extra={"hardware": run_config["gpu"], "fewshot_ids": [r["id"] for r in fewshot_rows]})
    (OUT / "base_eval.json").write_text((OUT / "base.json").read_text())
    for k, v in base.items():
        m = v["metrics"]
        print(f"BASE {k:10s} intent={m['intent_accuracy']} json={m['json_validity_rate']} item_f1={m['item_f1']}")
    del base_model; torch.cuda.empty_cache()

## 12. (Optional) Push the adapter to the Hugging Face Hub with a version tag

In [ ]:
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

token = get_hf_token() if PUSH_TO_HUB else None
if token and "your-hf-user" not in HF_MODEL_REPO:
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    api.create_repo(HF_MODEL_REPO, exist_ok=True, repo_type="model")
    card = f'''---
base_model: google/gemma-3-270m-it
library_name: peft
license: gemma
tags: [quotedesk, lora, extraction, synthetic-data]
---
# QuoteDesk student {VERSION_TAG}

LoRA adapter for Gemma 3 270M that extracts structured quote requests from messy customer emails
(fictional company, synthetic data). Trained with Unsloth; eval results in `eval.json`.

Gemma is provided under and subject to the Gemma Terms of Use found at ai.google.dev/gemma/terms.
Use of this adapter is subject to the Gemma Prohibited Use Policy (ai.google.dev/gemma/prohibited_use_policy).
'''
    (OUT / "adapter" / "README.md").write_text(card)
    (OUT / "adapter" / "NOTICE").write_text("Gemma is provided under and subject to the Gemma Terms of Use found at ai.google.dev/gemma/terms\n")
    (OUT / "adapter" / "GEMMA_TERMS.md").write_text("See https://ai.google.dev/gemma/terms and https://ai.google.dev/gemma/prohibited_use_policy\n"
                                                    "This adapter is a Model Derivative of Gemma 3 and is distributed under the Gemma Terms of Use.\n")
    for f in ("eval.json", "run_config.json", "loss_curve.png", "base_eval.json"):
        if (OUT / f).exists():
            (OUT / "adapter" / f).write_bytes((OUT / f).read_bytes())
    api.upload_folder(repo_id=HF_MODEL_REPO, folder_path=str(OUT / "adapter"),
                      commit_message=f"QuoteDesk student {VERSION_TAG}")
    try:
        api.delete_tag(HF_MODEL_REPO, tag=VERSION_TAG)
    except Exception:
        pass
    api.create_tag(HF_MODEL_REPO, tag=VERSION_TAG, tag_message=f"QuoteDesk student {VERSION_TAG}")
    print(f"pushed https://huggingface.co/{HF_MODEL_REPO} tag {VERSION_TAG}")
else:
    print("skipped push (set PUSH_TO_HUB, HF_MODEL_REPO and the HF_TOKEN secret)")

## Done
Download `/kaggle/working` outputs (or use the HF repo). Give the backend `HF_MODEL_REPO` and `STUDENT_ADAPTER_REVISION`
(the tag above), and copy `eval.json` / `base_eval.json` into `eval/reports/<timestamp>/` locally (see README_MANUAL.md).